In [ ]:
# Parameters
script_number = "045"

In [ ]:
import pdfplumber
import csv
import pandas as pd
from settings import page_settings, additional_speakers, ignore_line_markers
settings = page_settings[script_number]  # <- fill in script number
print(settings)
print(additional_speakers)
print(ignore_line_markers)

In [ ]:
PATH_TO_INPUT_PDF = f'../scripts/skript_{settings['id']}.pdf'
PATH_TO_OUTPUT_CSV = f'../csv/{settings['id']}.csv'

# Create a list of Speaker/Roles

In [ ]:
speakers = pd.read_csv('../data/rolle.tsv', sep='\t')
speakers = speakers.drop(columns='rolleID')
speakers = pd.concat([speakers, pd.Series(additional_speakers, name="name")], ignore_index=True)
speakers = speakers.reset_index(drop=True)
speakers

In [ ]:
speakers_lookup_set = set(speakers['name'].values)
def lookup_speaker_name(text: str):
    words = text.strip().split(' ')
    #print(words)
    matches = []
    for i in range(len(words)):
        #print(i)
        search_string = ' '.join(words[0:i+1])
        #print(search_string)
        if search_string in speakers_lookup_set:
            # append found word and last index of word in string
            matches.append((search_string, text.find(search_string) + len(search_string) - 1))
    if len(matches) == 0:
        return None, 0
    return max(matches, key=lambda x: len(x[0]))

In [ ]:
# test lookup
texts = ['Bob Andrews bla bla bla', 'Justus Jonas bli bla blup', '3???/ Gus Danke!']
for text in texts:
    print(text, lookup_speaker_name(text))

# Simple Text Extractor using pdfplumber
- extracts text while keeping layout
- iterates through text to find the layout
- needs manual settings for table layout

## Import and load PDF

In [ ]:
pdf = pdfplumber.open(PATH_TO_INPUT_PDF)

## Set Speaker Name Window

In [ ]:
# SETTINGS
SPEAKER_NAME_WINDOW = settings['speaker_name_window']

#SPEAKER_NAME_WINDOW = [5, 18]
#         |           WINDOW        |
#_._._._._|_._.J.U.S.T.U.S._._._.H.e|y.,._.w.o.....
#_._._._._|_._.B.L.A.C.K.B.E.A.R.D._|_.A.r.r.r.....
#_._._._._|_._._._._._._._._._._.W.o|_.i.s.t._.....
# Sometimes the speaker of a line is too long and pdfplumber doesnt visualise the table column correctly
# this leads to 'Blackbeards' name to overlap with the text line of the previous line
# SPEAKER_NAME_WINDOW is the start and end index where the program should look for the character name
# this should be as tiny and on point as possible
# the window should start in the whitespace area before the speaker name, but can end in the text area after the speaker

In [ ]:
TEST_SPEAKER_NAME_WINDOW = [5, 18]
print('       JUSTUS   Hey, wo'[TEST_SPEAKER_NAME_WINDOW[0]:TEST_SPEAKER_NAME_WINDOW[1]+1])
print('       BLACKBEARD  Arrr'[TEST_SPEAKER_NAME_WINDOW[0]:TEST_SPEAKER_NAME_WINDOW[1]+1])
print('                Wo ist '[TEST_SPEAKER_NAME_WINDOW[0]:TEST_SPEAKER_NAME_WINDOW[1]+1])

## Set Text Start Index

In [ ]:
TEXT_START_INDEX = settings['text_start_index']
# this is the index of the char where the speakers text already started
# this should be BEFORE the text starts

#SPEAKER_NAME_WINDOW = [5, 18]
#_._._._._|_._.J.U.S.T.U.S._._._.H.e|y.,._.w.o.....
#_._._._._|_._.B.L.A.C.K.B.E.A.R.D._|_.A.r.r.r.....
#_._._._._|_._._._._._._._._._._.W.o|_.i.s.t._.....
#                           ^
#     TEXT START INDEX      |

## Set Speaker Name Window Threshold
- look at the defined speaker name window of a text line
- count the number of whitespace and non-whitespace chararcters
- divide to get a percentage
- below threshold -> probably no speaker, just remnants of the text
- above threshold -> probably a speaker with some whitespaces at the start (remember to set the window wider at the start)

In [ ]:
SPEAKER_NAME_WINDOW_THRESHOLD = settings['speaker_name_window_threshold']

## Example for page 1

In [ ]:
page_text = pdf.pages[0].extract_text(layout=True)
print(page_text)

## Concat all pages

In [ ]:
extracted_raw_text = ''
for page in pdf.pages:
    extracted_raw_text += '\n' + f"PAGE {page.page_number}"
    page_text = page.extract_text(layout=True)
    extracted_raw_text += page_text 
print(extracted_raw_text)

## Spit texts to individual lines

In [ ]:
extracted_lines = extracted_raw_text.split('\n')
len(extracted_lines)

## Algorithm

In [ ]:
csv_rows = []  # rows for final csv (speaker, text, page)
current_speaker = None  # currently speaking
current_page_number = "0"

def save_speaker_and_text_line(speaker, text):
    csv_rows.append((speaker, text, current_page_number))

def save_only_text_line(line: str):
    text = line.strip()
    if current_speaker is None:
        csv_rows.append(('REGIE', text, current_page_number))
    else:
        csv_rows.append((current_speaker, text, current_page_number))

def ignore_line(line: str) -> bool:
    for marker in ignore_line_markers:
        if marker in line:
            return True
    return False

In [ ]:
csv_rows = []  # rows for final csv
current_speaker = None  # currently speaking
current_page_number = "0"

for line in extracted_lines:
    stripped_line = line.strip()

    if stripped_line == '' or ignore_line(stripped_line):
        # empty row or credit row -> speaker ends -> reset for new speaker
        #current_speaker = None
        continue

    if stripped_line.startswith('PAGE'):
        # detect previously inserted page number string (PAGE X) at top of each page
        page_number_str = stripped_line.replace('PAGE ', '')
        current_page_number = str(int(page_number_str))
        continue

    if stripped_line == current_page_number:
        # detect page numbers
        #print('Found page number', stripped_line)
        continue

    # try to find speaker in lookup table
    speaker_name_window_text = line[SPEAKER_NAME_WINDOW[0]:SPEAKER_NAME_WINDOW[1]+1]
    if len(speaker_name_window_text.strip()) == 0:
        # empty speaker name window -> only text in this line
        #print('Empty speaker name window')
        save_only_text_line(line)
        continue
    
    whitespace_percentage_for_speaker_window = len(speaker_name_window_text.strip()) / len(speaker_name_window_text)
    
    if whitespace_percentage_for_speaker_window < SPEAKER_NAME_WINDOW_THRESHOLD:
        # too much whitespace in speaker name window -> only text in this line
        #print('too much whitespace in speaker name window', line)
        save_only_text_line(line)
        continue
    
    speaker_name, speaker_name_last_idx = lookup_speaker_name(speaker_name_window_text)

    if speaker_name is not None:
        # Speaker found, this is a Speaker AND Text line
        text = line[SPEAKER_NAME_WINDOW[0]+speaker_name_last_idx+1:].strip()
        #if current_speaker is not None:
            # a new speaker is starting to say something, save the old and start new
            #save_to_rows()
        current_speaker = speaker_name.strip()
        save_speaker_and_text_line(current_speaker, text)
        #print('found speaker: ', current_speaker)
    else:
        print("Ist das ein Rollenname?: ", line)  # this should print lines where there should be a character but we found non in the lookup table
        # We detected a line with no Speaker and only Text
        save_only_text_line(line)

#save_to_rows()

In [ ]:
csv_rows

## Cleaning I
- pdfplumber actually gives us a lot of extra information about the text, like Font, Size, etc.
- We can leverage that by looking for italic/cursive texts which is typically used for REGIE interrupts
- in this step we create a small list of text bits that are written in italic in the text
- these text bits can then be removed or marked

In [ ]:
info = []
for page in pdf.pages:
    page_info = page.extract_text_lines(extra_attrs=["fontname", "size"])
    info.append(page_info)
info

In [ ]:
italic_lines = dict()
for page in pdf.pages:
    page_info = page.extract_text_lines(extra_attrs=["fontname", "size"])
    for line_info in page_info:
        # line_info has information about the text line (e.g. the line string and the containing chars)
        italic_in_this_line = []
        for char_info in line_info['chars']:
            # char_info has information about the chars (e.g. char font, char size, char position, etc.)
            if char_info['fontname'] == 'Arial-ItalicMT' or char_info['fontname'] == 'Arial-BoldItalicMT':
                italic_in_this_line.append(char_info['text'])
        if len(italic_in_this_line) > 0:
            italic_lines[line_info['text']] = italic_in_this_line

italic_texts = []

for key, value in italic_lines.items():
    target = ''.join(value)
    #print(key, '|', target)

    matches = []

    n = len(key)
    for i in range(n):
        for j in range(i + 1, n + 1):
            substr = key[i:j]
            if substr.replace(" ", "") == target and len(substr.strip()) > 2:
                # only accept matches that are larger than 2 (too skip punctuation, etc.)
                matches.append(substr.strip())
    
    if len(matches) == 0:
        print(key, value)
    else: 
        # remove duplicates while keeping order
        matches = list(dict.fromkeys(matches))[0]
        italic_texts.append(matches)

italic_texts

### Now lets use this list to clean some speakers

In [ ]:
import copy
csv_rows_corrected = copy.deepcopy(csv_rows)
corrected_lines = []

for idx, row in enumerate(csv_rows):
    speaker = row[0]
    text = row[1]
    if speaker != 'REGIE' and text.strip() in italic_texts:
        csv_rows_corrected[idx] = ('REGIE', row[1], row[2])
        corrected_lines.append(row)
print(f'Found {len(corrected_lines)} lines that were wrongly assigned to a speaker. I corrected them to "REGIE"')
corrected_lines

## Save to CSV

In [ ]:
with open(PATH_TO_OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f, quoting=csv.QUOTE_ALL)
    writer.writerow(['Speaker', 'Text', 'Page'])
    writer.writerows(csv_rows_corrected)